In [32]:
# %%
"""
================================================================================
  Flash Attention Applied to Pretrained ViT (CIFAR-10)
================================================================================

OVERVIEW
--------
This script loads the saved ViT baseline model and replaces its standard
self-attention with Flash Attention — WITHOUT changing the architecture,
weights, or any other component.

  Baseline      : vit_small_patch4 (32×32, pretrained ImageNet-21k)
                  Loaded from "__4__baseline_vit_pretrained_cifar10.pth"
  Flash-ViT     : Same model, same weights, same structure
                  ONLY the attention computation kernel is swapped

KEY PRINCIPLE
-------------
Flash Attention is orthogonal to architectural design.
It does not change:
  - The mathematical result:  softmax(QKᵀ/√d) · V  (identical output)
  - The model weights
  - The number of parameters
  - The model structure

It ONLY changes HOW attention is computed:
  - Tiles QKᵀ into blocks that fit on-chip SRAM
  - Never materialises the full N×N attention matrix in HBM
  - Computes softmax incrementally (online normalisation)
  - Reduces HBM memory reads/writes from O(N²) to O(N²/B)

================================================================================
MATHEMATICAL EQUIVALENCE
================================================================================

Standard attention (baseline):
  S   = Q @ Kᵀ / √d                  ← full N×N matrix, stored in HBM
  P   = softmax(S)                    ← full N×N softmax
  O   = P @ V                         ← output

Flash Attention (IO-aware tiling):
  For tile (i,j) of size Br × Bc:
    S_ij = Q_i @ K_jᵀ / √d           ← partial block scores (on-chip)
    m_i  = max(m_i_prev, rowmax(S_ij))
    ℓ_i  = e^(m_i_prev − m_i) · ℓ_i_prev + rowsum(e^(S_ij − m_i))
    O_i  = e^(m_i_prev − m_i) · O_i_prev + e^(S_ij − m_i) @ V_j
  Final: O_i = O_i / ℓ_i             ← mathematically identical to standard

Memory: O(N²) HBM → O(N · d) HBM  (the N×N matrix never leaves SRAM)

================================================================================
IMPLEMENTATION STRATEGY
================================================================================

PyTorch ≥ 2.0 exposes Flash Attention natively via:
  torch.nn.functional.scaled_dot_product_attention(Q, K, V)

When running on CUDA with compatible hardware, PyTorch automatically
dispatches to FlashAttention-2 or efficient_attention kernels.
On CPU or older GPUs it falls back to the standard math kernel
(still numerically correct, same output).

We replace timm's attention module by subclassing and overriding
the forward method — the weight tensors (qkv, proj) are kept identical.
No retraining needed; we just load the saved weights into the new module.

================================================================================
OUTPUT
--------
  flash_attention_metrics.json   — comparison metrics (standard vs flash)
================================================================================
"""

import os, json, time, warnings, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import timm
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score)

warnings.filterwarnings("ignore")

# ── Reproducibility ────────────────────────────────────────────────────────────
import random
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

# ── Config ─────────────────────────────────────────────────────────────────────
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE      = 128
NUM_CLASSES     = 10
IMG_SIZE        = 32
PATCH_SIZE      = 4
BASELINE_WEIGHTS = "../__4__baseline_vit_pretrained_cifar10.pth"
OUTPUT_JSON      = "flash_attention_metrics.json"
INFERENCE_RUNS   = 200

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)
CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog',      'frog',      'horse', 'ship', 'truck']

print(f"\n{'='*65}")
print("  Flash Attention on Pretrained ViT (CIFAR-10)")
print(f"  Device  : {DEVICE}")
print(f"  PyTorch : {torch.__version__}")
fa_available = (
    hasattr(F, "scaled_dot_product_attention") and
    torch.cuda.is_available()
)
print(f"  Flash Attention natively available: {fa_available}")
print(f"{'='*65}\n")


  Flash Attention on Pretrained ViT (CIFAR-10)
  Device  : cuda
  PyTorch : 2.12.0.dev20260408+cu128
  Flash Attention natively available: True



In [33]:
# =============================================================================
# Flash Attention Module
# =============================================================================

class FlashAttention(nn.Module):
    """
    Drop-in replacement for timm's Attention block.

    Mathematical operation is IDENTICAL to standard self-attention:
        O = softmax(QKᵀ / √d) · V

    The only difference is HOW it is computed:
        torch.nn.functional.scaled_dot_product_attention()
        dispatches to FlashAttention-2 on CUDA (Ampere+),
        or the efficient CUDA kernel on older GPUs,
        or the reference math implementation on CPU.

    All weight tensors (qkv projection, output projection, norms)
    are preserved exactly from the original timm attention block.
    """

    def __init__(self, timm_attn_block):
        """
        Initialise from an existing timm Attention block.
        Copies all weight references — no new parameters are created.
        """
        super().__init__()
        # Copy all sub-modules from the original timm attention block
        # This preserves weights exactly: qkv Linear, proj Linear, etc.
        self.qkv       = timm_attn_block.qkv       # (3*dim, dim) linear
        self.proj      = timm_attn_block.proj       # output projection
        self.num_heads = timm_attn_block.num_heads
        self.scale     = timm_attn_block.scale      # 1 / sqrt(head_dim)

        # attn_drop and proj_drop may or may not exist in timm versions
        self.attn_drop = getattr(timm_attn_block, "attn_drop", nn.Identity())
        self.proj_drop = getattr(timm_attn_block, "proj_drop", nn.Identity())

        # Store dropout probability for SDPA interface
        attn_drop_p = 0.0
        if hasattr(self.attn_drop, "p"):
            attn_drop_p = self.attn_drop.p
        self.attn_drop_p = attn_drop_p

    def forward(self, x, attn_mask=None, is_causal=False):
        """
        Forward pass using torch.nn.functional.scaled_dot_product_attention.

        IO-Aware computation (when dispatched to FlashAttention kernel):
          - Q, K, V tiles computed on-chip (SRAM)
          - Full N×N attention matrix NEVER written to HBM
          - Softmax computed incrementally via online normalisation
          - Memory: O(N²) HBM → O(N·d) HBM

        On CPU or older GPUs: falls back to the standard math kernel
        with identical numerical output.
        """
        B, N, C = x.shape
        H = self.num_heads
        head_dim = C // H

        # Project to Q, K, V  (same linear layer as original)
        qkv = (self.qkv(x)
                   .reshape(B, N, 3, H, head_dim)
                   .permute(2, 0, 3, 1, 4))         # (3, B, H, N, head_dim)
        q, k, v = qkv.unbind(0)                     # each: (B, H, N, head_dim)

        # ── Flash Attention dispatch ──────────────────────────────────────────
        # torch.nn.functional.scaled_dot_product_attention:
        #   - CUDA + Ampere+  → FlashAttention-2 kernel (tiled, IO-aware)
        #   - CUDA older GPU  → Efficient attention kernel
        #   - CPU             → Standard math (reference implementation)
        # All three paths compute:  softmax(Q @ Kᵀ / √d) @ V
        # ─────────────────────────────────────────────────────────────────────
        dropout_p = self.attn_drop_p if self.training else 0.0
        x = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            is_causal=is_causal,
            dropout_p=dropout_p,
            scale=self.scale,            # uses stored scale (= 1/√head_dim)
        )                                # output: (B, H, N, head_dim)

        # Merge heads and apply output projection (same weights as original)
        x = x.transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


def inject_flash_attention(model: nn.Module) -> nn.Module:
    """
    Walk the model and replace every timm Attention block with FlashAttention.

    This is a WEIGHT-PRESERVING operation:
      - No parameters are added or removed
      - No weights are modified
      - The model is NOT retrained
      - Only the forward() computation path changes

    Returns the modified model (in-place modification + return for chaining).
    """
    replaced = 0
    for name, module in model.named_children():
        # Recursively descend into sub-modules
        if len(list(module.children())) > 0:
            inject_flash_attention(module)

        # Target: timm's Attention class (identified by attribute signature)
        # We check for the characteristic attributes rather than isinstance()
        # so this works across timm versions.
        if (hasattr(module, "qkv") and
                hasattr(module, "proj") and
                hasattr(module, "num_heads") and
                not isinstance(module, FlashAttention)):
            setattr(model, name, FlashAttention(module))
            replaced += 1

    return model

In [34]:
# =============================================================================
# Model builder (mirrors baseline script exactly)
# =============================================================================

def build_baseline_vit(num_classes=NUM_CLASSES):
    """Recreate the exact same ViT architecture used in the baseline script."""
    model_pretrained = timm.create_model(
        'vit_small_patch16_224', pretrained=False, num_classes=num_classes)

    model = timm.create_model(
        'vit_small_patch16_224',
        pretrained=False,
        num_classes=num_classes,
        img_size=IMG_SIZE,
        patch_size=PATCH_SIZE,
    )
    return model


In [35]:
# =============================================================================
# Data loader
# =============================================================================

def get_test_loader():
    tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(
        "../data", train=False, download=True, transform=tf)
    return torch.utils.data.DataLoader(
        ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=0, pin_memory=True)

In [36]:
# =============================================================================
# Evaluation helpers
# =============================================================================

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        preds.extend(model(x).argmax(1).cpu().numpy())
        labels.extend(y.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred,
                                           average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred,
                                        average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred,
                                    average="macro", zero_division=0)),
    }


def measure_latency_cpu(model):
    """
    CPU timing with time.perf_counter() — mirrors reference script exactly.
    Single image: 10 warm-up + 100 measured runs.
    Batch-128   :  5 warm-up +  20 measured runs.
    """
    model_cpu = model.cpu().eval()

    dummy_single = torch.randn(1,   3, IMG_SIZE, IMG_SIZE)
    dummy_batch  = torch.randn(128, 3, IMG_SIZE, IMG_SIZE)

    # Single image
    with torch.no_grad():
        for _ in range(10):
            model_cpu(dummy_single)
    times_single = []
    with torch.no_grad():
        for _ in range(100):
            t0 = time.perf_counter()
            model_cpu(dummy_single)
            times_single.append(time.perf_counter() - t0)
    single_ms = np.mean(times_single) * 1000

    # Batch 128
    with torch.no_grad():
        for _ in range(5):
            model_cpu(dummy_batch)
    times_batch = []
    with torch.no_grad():
        for _ in range(20):
            t0 = time.perf_counter()
            model_cpu(dummy_batch)
            times_batch.append(time.perf_counter() - t0)
    batch_ms   = np.mean(times_batch) * 1000
    per_img_ms = batch_ms / 128
    throughput = 128 / (batch_ms / 1000)

    # Move back to original device after CPU measurement
    model.to(DEVICE)

    return {
        "single_img_ms"      : round(single_ms,  4),
        "batch128_ms"        : round(batch_ms,    4),
        "per_img_ms"         : round(per_img_ms,  4),
        "throughput_imgs_sec": round(throughput,  1),
        "timing_method"      : "time.perf_counter()",
    }


def measure_latency_gpu(model):
    """
    GPU timing — mirrors reference script exactly:
      Single image : torch.cuda.synchronize() + perf_counter, 50 warm-up + 500 runs
      Batch 128    : CUDA Events,                              10 warm-up + 100 runs
    """
    model = model.to(DEVICE).eval()

    dummy_single = torch.randn(1,   3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    dummy_batch  = torch.randn(128, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

    # Single image — synchronized perf_counter
    with torch.no_grad():
        for _ in range(50):
            model(dummy_single)
    torch.cuda.synchronize()

    times_single = []
    with torch.no_grad():
        for _ in range(500):
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(dummy_single)
            torch.cuda.synchronize()
            times_single.append(time.perf_counter() - t0)
    single_ms = np.mean(times_single) * 1000

    # Batch 128 — CUDA events
    start_ev = torch.cuda.Event(enable_timing=True)
    end_ev   = torch.cuda.Event(enable_timing=True)

    with torch.no_grad():
        for _ in range(10):
            model(dummy_batch)
    torch.cuda.synchronize()

    batch_times = []
    with torch.no_grad():
        for _ in range(100):
            start_ev.record()
            model(dummy_batch)
            end_ev.record()
            torch.cuda.synchronize()
            batch_times.append(start_ev.elapsed_time(end_ev))

    batch_ms   = np.mean(batch_times)
    per_img_ms = batch_ms / 128
    throughput = 128 / (batch_ms / 1000)

    return {
        "single_img_ms"      : round(single_ms,  4),
        "batch128_ms"        : round(batch_ms,    4),
        "per_img_ms"         : round(per_img_ms,  4),
        "throughput_imgs_sec": round(throughput,  1),
        "timing_method"      : "CUDA events + torch.cuda.synchronize()",
    }


def param_count(model):
    return sum(p.numel() for p in model.parameters())


def count_attention_blocks(model):
    """Count how many attention blocks are in the model."""
    n_standard = sum(1 for _, m in model.named_modules()
                     if hasattr(m, "qkv") and hasattr(m, "num_heads")
                     and not isinstance(m, FlashAttention))
    n_flash    = sum(1 for _, m in model.named_modules()
                     if isinstance(m, FlashAttention))
    return n_standard, n_flash

In [37]:
# =============================================================================
# Main
# =============================================================================

def main():
    print(f"Loading test set...")
    test_loader = get_test_loader()

    # ─────────────────────────────────────────────────────────────────────────
    # 1. Standard ViT — load saved weights, no modification
    # ─────────────────────────────────────────────────────────────────────────
    print(f"\n{'─'*55}")
    print("  1. Standard ViT (baseline, unmodified)")
    print(f"{'─'*55}")

    standard_model = build_baseline_vit().to(DEVICE)
    state_dict = torch.load(BASELINE_WEIGHTS, map_location=DEVICE)
    standard_model.load_state_dict(state_dict)
    standard_model.eval()

    n_std, n_fl = count_attention_blocks(standard_model)
    print(f"  Attention blocks   : {n_std} standard  |  {n_fl} flash")
    print(f"  Parameters         : {param_count(standard_model)/1e6:.3f} M")

    print("  Evaluating accuracy...")
    std_metrics = evaluate(standard_model, test_loader)
    print(f"  Accuracy           : {std_metrics['accuracy']:.4f}")

    print("  Measuring CPU latency...")
    std_latency_cpu = measure_latency_cpu(standard_model)
    print(f"  Single-img CPU     : {std_latency_cpu['single_img_ms']:.3f} ms")
    print(f"  Batch-128 CPU      : {std_latency_cpu['batch128_ms']:.2f} ms")
    print(f"  Throughput CPU     : {std_latency_cpu['throughput_imgs_sec']:.1f} imgs/s")
    print("  Measuring GPU latency...")
    std_latency_gpu = measure_latency_gpu(standard_model)
    print(f"  Single-img GPU     : {std_latency_gpu['single_img_ms']:.3f} ms")
    print(f"  Batch-128 GPU      : {std_latency_gpu['batch128_ms']:.2f} ms")
    print(f"  Throughput GPU     : {std_latency_gpu['throughput_imgs_sec']:.1f} imgs/s")

    # ─────────────────────────────────────────────────────────────────────────
    # 2. Flash-ViT — same weights, attention computation replaced
    # ─────────────────────────────────────────────────────────────────────────
    print(f"\n{'─'*55}")
    print("  2. Flash-ViT (same weights, Flash Attention kernel)")
    print(f"{'─'*55}")

    # Deep-copy to avoid modifying the standard model in-place
    flash_model = copy.deepcopy(standard_model)

    # Inject Flash Attention: replaces timm Attention → FlashAttention
    # Weight tensors are SHARED (not copied), so no extra memory is used.
    inject_flash_attention(flash_model)
    flash_model.eval()

    n_std_after, n_fl_after = count_attention_blocks(flash_model)
    print(f"  Attention blocks   : {n_std_after} standard  |  {n_fl_after} flash")
    print(f"  Parameters         : {param_count(flash_model)/1e6:.3f} M  "
          f"(unchanged by design)")

    print("  Evaluating accuracy (should be identical to baseline)...")
    fl_metrics = evaluate(flash_model, test_loader)
    acc_delta  = abs(fl_metrics["accuracy"] - std_metrics["accuracy"])
    print(f"  Accuracy           : {fl_metrics['accuracy']:.4f}  "
          f"(Δ = {acc_delta:.6f} — expected ≈ 0)")

    print("  Measuring CPU latency...")
    fl_latency_cpu = measure_latency_cpu(flash_model)
    print(f"  Single-img CPU     : {fl_latency_cpu['single_img_ms']:.3f} ms")
    print(f"  Batch-128 CPU      : {fl_latency_cpu['batch128_ms']:.2f} ms")
    print(f"  Throughput CPU     : {fl_latency_cpu['throughput_imgs_sec']:.1f} imgs/s")
    print("  Measuring GPU latency...")
    fl_latency_gpu = measure_latency_gpu(flash_model)
    print(f"  Single-img GPU     : {fl_latency_gpu['single_img_ms']:.3f} ms")
    print(f"  Batch-128 GPU      : {fl_latency_gpu['batch128_ms']:.2f} ms")
    print(f"  Throughput GPU     : {fl_latency_gpu['throughput_imgs_sec']:.1f} imgs/s")

    # ─────────────────────────────────────────────────────────────────────────
    # 3. Verify mathematical equivalence (single batch, exact match check)
    # ─────────────────────────────────────────────────────────────────────────
    print(f"\n{'─'*55}")
    print("  3. Numerical equivalence check (single batch)")
    print(f"{'─'*55}")

    x_test, _ = next(iter(test_loader))
    x_test = x_test.to(DEVICE)

    standard_model.eval()
    flash_model.eval()

    with torch.no_grad():
        out_std   = standard_model(x_test)
        out_flash = flash_model(x_test)

    max_diff = (out_std - out_flash).abs().max().item()
    mean_diff= (out_std - out_flash).abs().mean().item()
    print(f"  Max  logit diff    : {max_diff:.2e}   (float32 numerics)")
    print(f"  Mean logit diff    : {mean_diff:.2e}")
    print(f"  Outputs identical  : {max_diff < 1e-3}")
    print("  Note: small fp32 differences are expected from kernel rounding")

    # ─────────────────────────────────────────────────────────────────────────
    # 4. Speed-up summary (GPU batch, matching reference script convention)
    # ─────────────────────────────────────────────────────────────────────────
    speedup_single_gpu = std_latency_gpu["single_img_ms"] / fl_latency_gpu["single_img_ms"]
    speedup_batch_gpu  = std_latency_gpu["batch128_ms"]   / fl_latency_gpu["batch128_ms"]
    speedup_single_cpu = std_latency_cpu["single_img_ms"] / fl_latency_cpu["single_img_ms"]
    speedup_batch_cpu  = std_latency_cpu["batch128_ms"]   / fl_latency_cpu["batch128_ms"]
    throughput_gain_gpu = (fl_latency_gpu["throughput_imgs_sec"] /
                           std_latency_gpu["throughput_imgs_sec"] - 1) * 100
    throughput_gain_cpu = (fl_latency_cpu["throughput_imgs_sec"] /
                           std_latency_cpu["throughput_imgs_sec"] - 1) * 100

    # ─────────────────────────────────────────────────────────────────────────
    # 5. Save JSON — same structure as __4__baseline_metrics_v3.json
    # ─────────────────────────────────────────────────────────────────────────
    def _inference_block(lat_cpu, lat_gpu):
        """Build the inference_ms sub-dict in the exact reference format."""
        return {
            "cpu": {
                "single_img_ms"      : lat_cpu["single_img_ms"],
                "batch128_ms"        : lat_cpu["batch128_ms"],
                "per_img_ms"         : lat_cpu["per_img_ms"],
                "throughput_imgs_sec": lat_cpu["throughput_imgs_sec"],
                "timing_method"      : lat_cpu["timing_method"],
            },
            "gpu": {
                "single_img_ms"      : lat_gpu["single_img_ms"],
                "batch128_ms"        : lat_gpu["batch128_ms"],
                "per_img_ms"         : lat_gpu["per_img_ms"],
                "throughput_imgs_sec": lat_gpu["throughput_imgs_sec"],
                "timing_method"      : lat_gpu["timing_method"],
            },
        }

    report = {
        "method"     : "flash_attention",
        "description": (
            "Flash Attention applied to a pretrained ViT "
            "(vit_small patch4 32×32). Same weights, same architecture, "
            "only the attention computation kernel is replaced."
        ),
        "key_properties": {
            "mathematically_equivalent": True,
            "weights_modified"         : False,
            "retraining_required"      : False,
            "architecture_changed"     : False,
            "attention_blocks_replaced": n_fl_after,
        },
        "flash_attention_mechanism": {
            "standard_complexity": "O(N^2) HBM reads/writes for N×N attention matrix",
            "flash_complexity"   : "O(N^2/B) HBM reads/writes — N×N matrix stays in SRAM",
            "pytorch_dispatch"   : "torch.nn.functional.scaled_dot_product_attention",
            "kernel_on_ampere"   : "FlashAttention-2",
            "kernel_on_older_gpu": "Efficient CUDA attention kernel",
            "kernel_on_cpu"      : "Standard math (reference, numerically identical)",
            "softmax"            : "Incremental online normalisation (numerically stable)",
        },
        # ── Standard ViT (reference) — mirrors baseline_metrics_v3.json ──────
        "standard_vit": {
            "model"          : "vit_small_patch4_32_pretrained",
            "accuracy"       : std_metrics["accuracy"],
            "precision"      : std_metrics["precision"],
            "recall"         : std_metrics["recall"],
            "f1"             : std_metrics["f1"],
            "params"         : param_count(standard_model),
            "size_mb"        : round(os.path.getsize(BASELINE_WEIGHTS) / 1e6, 4),
            "input_resolution": IMG_SIZE,
            "patch_size"     : PATCH_SIZE,
            "num_patches"    : 64,
            "attention_type" : "standard (full N×N matrix materialised in HBM)",
            "inference_ms"   : _inference_block(std_latency_cpu, std_latency_gpu),
        },
        # ── Flash-ViT — same fields, same layout ──────────────────────────────
        "flash_vit": {
            "model"          : "vit_small_patch4_32_flash_attention",
            "accuracy"       : fl_metrics["accuracy"],
            "precision"      : fl_metrics["precision"],
            "recall"         : fl_metrics["recall"],
            "f1"             : fl_metrics["f1"],
            "params"         : param_count(flash_model),
            "size_mb"        : round(os.path.getsize(BASELINE_WEIGHTS) / 1e6, 4),
            "input_resolution": IMG_SIZE,
            "patch_size"     : PATCH_SIZE,
            "num_patches"    : 64,
            "attention_type" : "flash (tiled, IO-aware, no N×N matrix in HBM)",
            "inference_ms"   : _inference_block(fl_latency_cpu, fl_latency_gpu),
        },
        # ── Comparison ────────────────────────────────────────────────────────
        "comparison": {
            "accuracy_delta"          : round(acc_delta, 8),
            "max_logit_diff"          : round(max_diff,  8),
            "mean_logit_diff"         : round(mean_diff, 8),
            "speedup_single_img_gpu"  : round(speedup_single_gpu, 4),
            "speedup_batch128_gpu"    : round(speedup_batch_gpu,  4),
            "throughput_gain_pct_gpu" : round(throughput_gain_gpu, 2),
            "speedup_single_img_cpu"  : round(speedup_single_cpu, 4),
            "speedup_batch128_cpu"    : round(speedup_batch_cpu,  4),
            "throughput_gain_pct_cpu" : round(throughput_gain_cpu, 2),
        },
        "notes": {
            "orthogonality": (
                "Flash Attention is orthogonal to architecture design. "
                "It can be combined with pruning, quantization, distillation, "
                "or NAS without any conflict."
            ),
            "memory_benefit": (
                "Memory savings grow quadratically with sequence length. "
                "For N=64 patches (CIFAR-10) the benefit is modest. "
                "For larger inputs (N=196 for 224×224) the gain is substantial."
            ),
            "speedup_note": (
                "Speedup depends on hardware (Ampere GPU sees most benefit), "
                "sequence length (longer → more gain), and batch size."
            ),
        },
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)
    print(f"\n✓ Metrics saved → {OUTPUT_JSON}")

    # ─────────────────────────────────────────────────────────────────────────
    # 6. Print summary table
    # ─────────────────────────────────────────────────────────────────────────
    W = 78
    print(f"\n{'='*W}")
    print(f"  {'Model':<25} {'Acc':>7} {'F1':>7} "
          f"{'CPU 1img':>9} {'CPU b128':>9} {'GPU 1img':>9} {'GPU b128':>9}")
    print(f"  {'─'*25} {'─'*7} {'─'*7} {'─'*9} {'─'*9} {'─'*9} {'─'*9}")
    print(f"  {'Standard ViT':<25} "
          f"{std_metrics['accuracy']:>7.4f} {std_metrics['f1']:>7.4f} "
          f"{std_latency_cpu['single_img_ms']:>8.3f}ms "
          f"{std_latency_cpu['batch128_ms']:>8.2f}ms "
          f"{std_latency_gpu['single_img_ms']:>8.3f}ms "
          f"{std_latency_gpu['batch128_ms']:>8.2f}ms")
    print(f"  {'Flash-ViT':<25} "
          f"{fl_metrics['accuracy']:>7.4f} {fl_metrics['f1']:>7.4f} "
          f"{fl_latency_cpu['single_img_ms']:>8.3f}ms "
          f"{fl_latency_cpu['batch128_ms']:>8.2f}ms "
          f"{fl_latency_gpu['single_img_ms']:>8.3f}ms "
          f"{fl_latency_gpu['batch128_ms']:>8.2f}ms")
    print(f"  {'─'*25} {'─'*7} {'─'*7} {'─'*9} {'─'*9} {'─'*9} {'─'*9}")
    print(f"  {'Speedup':<25} {'—':>7} {'—':>7} "
          f"{speedup_single_cpu:>8.3f}x "
          f"{speedup_batch_cpu:>8.3f}x "
          f"{speedup_single_gpu:>8.3f}x "
          f"{speedup_batch_gpu:>8.3f}x")
    print(f"{'='*W}\n")
    print(f"  Accuracy delta (Flash − Standard) : {acc_delta:.2e}  ≈ 0 ✓")
    print(f"  Attention blocks replaced         : {n_fl_after} / {n_fl_after}")
    print(f"  Parameters changed                : None ✓")


if __name__ == "__main__":
    main()

Loading test set...

───────────────────────────────────────────────────────
  1. Standard ViT (baseline, unmodified)
───────────────────────────────────────────────────────
  Attention blocks   : 12 standard  |  0 flash
  Parameters         : 21.342 M
  Evaluating accuracy...
  Accuracy           : 0.9666
  Measuring CPU latency...
  Single-img CPU     : 10.426 ms
  Batch-128 CPU      : 646.87 ms
  Throughput CPU     : 197.9 imgs/s
  Measuring GPU latency...
  Single-img GPU     : 4.700 ms
  Batch-128 GPU      : 55.16 ms
  Throughput GPU     : 2320.4 imgs/s

───────────────────────────────────────────────────────
  2. Flash-ViT (same weights, Flash Attention kernel)
───────────────────────────────────────────────────────
  Attention blocks   : 0 standard  |  12 flash
  Parameters         : 21.342 M  (unchanged by design)
  Evaluating accuracy (should be identical to baseline)...
  Accuracy           : 0.9666  (Δ = 0.000000 — expected ≈ 0)
  Measuring CPU latency...
  Single-img CPU   